In [ ]:
# iniciar notebook
from functions.Diretories_Downloads import dir_project, cif_download
from functions.Primary_Filter import primary_filter

In [ ]:
# CELL 1 (cria o diretório do projeto)
Project_Name = "projeto_hematita3M"
dir_, dir_ref = dir_project(Project_Name)

print(dir_)
print(dir_ref)

In [ ]:
# CELL 2 (ref download)

# COD_IDs de Referência - Óxidos de Ferro e Fases Relacionadas
# Hematita: 1011240
# Magnetita: 9002318
# Maghemita: 1528611
# Goethita: 9003077
# Wüstite: 1011119
# Ferro Metálico: 9006610

cod_id_list = [1011240, 9002318, 1528611, 9003077, 1011119, 9006610]
for cod_ID in cod_id_list:
    cif_download(cod_ID, dir_ref)

In [ ]:
#run Primary_Filter.py
input_drx_raw = "input/Magnetite_xrd.txt"
primary_filter(input_file=input_drx_raw, ref_dir=dir_ref)

In [ ]:
# Plot: Amostra vs Melhor Referência (CIF simulado)
import matplotlib.pyplot as plt
from functions.Primary_Filter import carregar_amostra, simular_padrao_xrd, listar_cif_para_dict
from scipy.stats import pearsonr
import numpy as np

# Carrega a amostra
theta, int_amostra = carregar_amostra(input_drx_raw)
amostra_norm = (int_amostra - int_amostra.min()) / int_amostra.max()

# Encontra o melhor match
candidatos = listar_cif_para_dict(dir_ref)
melhor_nome, melhor_score = None, -np.inf
for nome, cif_path in candidatos.items():
    try:
        int_sim = simular_padrao_xrd(cif_path, theta)
        ref_norm = (int_sim - int_sim.min()) / (int_sim.max() or 1.0)
        score, _ = pearsonr(amostra_norm, ref_norm)
        if score > melhor_score:
            melhor_nome, melhor_score, melhor_ref_norm = nome, score, ref_norm
    except Exception:
        pass

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(theta, amostra_norm, label="Amostra (experimental)", linewidth=0.8)
ax.plot(theta, melhor_ref_norm, label=f"Referência: {melhor_nome} (score={melhor_score:.2%})", linewidth=0.8, alpha=0.8)
ax.set_xlabel("2θ (°)")
ax.set_ylabel("Intensidade Normalizada")
ax.set_title("XRD: Amostra vs Melhor Referência")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
#run Workflow_GSAS-II.py
from functions.Workflow_GSAS2 import refinamento_sequencial_oxidos

arquivo_cif = "project/projeto_hematita3M/ref/cod_9002318.cif"
input_inst = "input/inst_rruff.prm"

refinamento_sequencial_oxidos(input_drx_raw, input_inst, arquivo_cif, Project_Name)